# EDA — Gym288 & Gym99
Skeleton action recognition datasets. All analysis focused on diagnosing training issues.

In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/schizocatto/Yolo-ST-GCN.git'
BRANCH   = 'refactor-1'
REPO_DIR = Path('/content/Yolo-ST-GCN') if IN_COLAB else Path(os.getcwd())

if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'huggingface_hub', 'scipy', 'scikit-learn'], check=True)
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', '-b', BRANCH, '--single-branch',
                        REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', 'origin', BRANCH], check=True)

os.chdir(str(REPO_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print('Working dir:', os.getcwd())

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})
SEED = 42
np.random.seed(SEED)

In [ ]:
# ── Paths ─────────────────────────────────────────────────────
if IN_COLAB:
    GYM288_PKL = '/content/Gym288-skeleton/gym_288_skeleton.pkl'
    GYM99_PKL  = '/content/Gym99-from-Gym288/gym99_from_gym288.pkl'
else:
    GYM288_PKL = 'data/Gym288-skeleton/gym_288_skeleton.pkl'
    GYM99_PKL  = 'data/Gym99-from-Gym288/gym99_from_gym288.pkl'

In [ ]:
# ── Download Gym288 (Colab only) ──────────────────────────────
if IN_COLAB and not Path(GYM288_PKL).exists():
    from huggingface_hub import snapshot_download
    dl = Path(GYM288_PKL).parent
    dl.mkdir(parents=True, exist_ok=True)
    snapshot_download(repo_id='Lozumi/Gym288-skeleton', repo_type='dataset',
                      local_dir=str(dl), local_dir_use_symlinks=False)
    GYM288_PKL = str(sorted(dl.rglob('*.pkl'))[0])
print('Gym288:', GYM288_PKL)

In [ ]:
# ── Load Gym288 ───────────────────────────────────────────────
with open(GYM288_PKL, 'rb') as f:
    raw288 = pickle.load(f)

anns288   = raw288['annotations']
split288  = raw288['split']
train_ids288 = set(split288.get('train', []))
test_ids288  = set(split288.get('test',  []))

labels288 = np.array([int(a['label'])      for a in anns288])
dirs288   = [str(a['frame_dir'])           for a in anns288]
flags288  = np.array([1 if d in train_ids288 else (0 if d in test_ids288 else -1)
                      for d in dirs288], dtype=np.int8)

frames288 = []
for a in anns288:
    kp = np.asarray(a['keypoint'])
    frames288.append(kp.shape[1] if kp.ndim == 4 else kp.shape[0])
frames288 = np.array(frames288)

print(f'Gym288  total={len(labels288)}  train={(flags288==1).sum()}  '
      f'test={(flags288==0).sum()}  unknown={(flags288==-1).sum()}')
print(f'Classes: {len(np.unique(labels288))}  Labels: {labels288.min()}–{labels288.max()}')

In [ ]:
# ── Build + Load Gym99 ────────────────────────────────────────
if not Path(GYM99_PKL).exists():
    from src.gym99_builder import build_gym99_from_gym288_pickle
    Path(GYM99_PKL).parent.mkdir(parents=True, exist_ok=True)
    stats = build_gym99_from_gym288_pickle(GYM288_PKL, GYM99_PKL)
    print('Build stats:', stats)

with open(GYM99_PKL, 'rb') as f:
    raw99 = pickle.load(f)

anns99   = raw99['annotations']
split99  = raw99['split']
train_ids99 = set(split99.get('train', []))
test_ids99  = set(split99.get('test',  []))

labels99 = np.array([int(a['label'])     for a in anns99])
dirs99   = [str(a['frame_dir'])          for a in anns99]
flags99  = np.array([1 if d in train_ids99 else (0 if d in test_ids99 else -1)
                     for d in dirs99], dtype=np.int8)

frames99 = []
for a in anns99:
    kp = np.asarray(a['keypoint'])
    frames99.append(kp.shape[1] if kp.ndim == 4 else kp.shape[0])
frames99 = np.array(frames99)

print(f'Gym99   total={len(labels99)}  train={(flags99==1).sum()}  '
      f'test={(flags99==0).sum()}  unknown={(flags99==-1).sum()}')
print(f'Classes: {len(np.unique(labels99))}  Labels: {labels99.min()}–{labels99.max()}')

---
## 1. Dataset Overview

In [ ]:
# Load processed tensors for skeleton analysis (coco18 layout)
from src.gym99_dataset import build_gym99_data_tensors

data99, labels99t, flags99t, _, _ = build_gym99_data_tensors(
    dataset_path=GYM99_PKL, joint_spec_name='coco18',
    split='all', keep_unknown_split=False,
)
# data99: (N, 2, T, 18, 1)  float32
print(f'Processed tensor: {data99.shape}  dtype={data99.dtype}')
print(f'Channels: 0=x  1=y  |  T={data99.shape[2]} frames  V={data99.shape[3]} joints')

In [ ]:
def gini(values):
    v = np.sort(np.array(values, dtype=float))
    n = len(v)
    idx = np.arange(1, n + 1)
    return float((2 * np.dot(idx, v)) / (n * v.sum()) - (n + 1) / n)

def effective_n(values):
    v = np.array(values, dtype=float)
    p = v / v.sum()
    return float(np.exp(-np.sum(p * np.log(p + 1e-12))))

counts288 = Counter(labels288.tolist())
counts99  = Counter(labels99.tolist())
vals288   = np.array(sorted(counts288.values(), reverse=True))
vals99    = np.array(sorted(counts99.values(),  reverse=True))

rows = []
for name, vals, labels, flags in [
    ('Gym288', vals288, labels288, flags288),
    ('Gym99',  vals99,  labels99,  flags99),
]:
    rows.append({
        'Dataset':          name,
        'Total':            len(labels),
        'Train':            int((flags == 1).sum()),
        'Test':             int((flags == 0).sum()),
        'Classes':          int(len(np.unique(labels))),
        'Min/class':        int(vals.min()),
        'Median/class':     int(np.median(vals)),
        'Max/class':        int(vals.max()),
        'Imbalance ratio':  f'{vals.max()/max(vals.min(),1):.0f}x',
        'Gini':             f'{gini(vals):.3f}',
        'Effective N':      f'{effective_n(vals):.1f} / {len(np.unique(labels))}',
    })

display(pd.DataFrame(rows).set_index('Dataset').T)

---
## 2. Class Distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for row, (name, vals, color) in enumerate([
    ('Gym288', vals288, 'steelblue'),
    ('Gym99',  vals99,  'coral'),
]):
    # Sorted class sizes (long-tail)
    ax = axes[row, 0]
    ax.bar(range(len(vals)), vals, color=color, width=1.0)
    ax.axhline(np.mean(vals), color='black', linestyle='--', linewidth=1,
               label=f'mean={np.mean(vals):.0f}')
    ax.set_xlabel('Class rank (sorted)')
    ax.set_ylabel('Samples')
    ax.set_title(f'{name} — class size (sorted)')
    ax.legend(fontsize=8)

    # Histogram of class sizes
    ax = axes[row, 1]
    ax.hist(vals, bins=30, color=color, edgecolor='white')
    ax.axvline(np.mean(vals), color='black', linestyle='--', linewidth=1,
               label=f'mean={np.mean(vals):.0f}')
    ax.set_xlabel('Samples per class')
    ax.set_ylabel('# classes')
    ax.set_title(f'{name} — class size histogram')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

for name, vals in [('Gym288', vals288), ('Gym99', vals99)]:
    print(f'{name}  Gini={gini(vals):.3f}  EffectiveN={effective_n(vals):.1f}/{len(vals)}'
          f'  <5samples: {(vals<5).sum()}  <10samples: {(vals<10).sum()}')

In [ ]:
# Lorenz curve (visual imbalance)
fig, ax = plt.subplots(figsize=(6, 5))

for name, vals, color in [('Gym288', vals288, 'steelblue'), ('Gym99', vals99, 'coral')]:
    v = np.sort(vals)
    cumulative = np.cumsum(v) / v.sum()
    x = np.linspace(0, 1, len(v))
    ax.plot(x, cumulative, label=f'{name} (Gini={gini(vals):.3f})', color=color, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Perfect equality')
ax.fill_between([0, 1], [0, 1], alpha=0.05, color='gray')
ax.set_xlabel('Cumulative fraction of classes')
ax.set_ylabel('Cumulative fraction of samples')
ax.set_title('Lorenz curve — class imbalance')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. Train / Test Split Quality

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (name, labels, flags, color) in zip(axes, [
    ('Gym288', labels288, flags288, 'steelblue'),
    ('Gym99',  labels99,  flags99,  'coral'),
]):
    classes = np.unique(labels)
    ratios  = []
    zero_test = 0
    for c in classes:
        tr = ((labels == c) & (flags == 1)).sum()
        te = ((labels == c) & (flags == 0)).sum()
        total = tr + te
        ratios.append(tr / total if total > 0 else 1.0)
        if te == 0:
            zero_test += 1

    ratios = np.array(ratios)
    ax.hist(ratios, bins=30, color=color, edgecolor='white')
    ax.axvline(np.mean(ratios), color='black', linestyle='--', linewidth=1,
               label=f'mean={np.mean(ratios):.2f}')
    ax.set_xlabel('Train fraction per class')
    ax.set_ylabel('# classes')
    ax.set_title(f'{name} — train fraction per class\n({zero_test} classes have zero test samples)')
    ax.legend(fontsize=8)
    print(f'{name}  zero-test classes: {zero_test}/{len(classes)}  '
          f'mean_train_frac={np.mean(ratios):.3f}  std={np.std(ratios):.3f}')

plt.tight_layout()
plt.show()

In [ ]:
# Data leakage check: same frame_dir in both train and test?
for name, train_ids, test_ids in [
    ('Gym288', train_ids288, test_ids288),
    ('Gym99',  train_ids99,  test_ids99),
]:
    overlap = train_ids & test_ids
    print(f'{name}  frame_dirs in both splits: {len(overlap)}  '
          + ('⚠ LEAKAGE' if overlap else '✓ clean'))

---
## 4. Temporal Analysis

In [ ]:
# Overall frame count distributions
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (name, fc, color) in zip(axes, [
    ('Gym288', frames288, 'steelblue'),
    ('Gym99',  frames99,  'coral'),
]):
    ax.hist(fc, bins=50, color=color, edgecolor='white')
    ax.axvline(64, color='red',   linestyle='--', linewidth=1.5, label='TARGET=64')
    ax.axvline(np.median(fc), color='black', linestyle=':', linewidth=1.5,
               label=f'median={int(np.median(fc))}')
    ax.set_xlabel('Raw frame count')
    ax.set_ylabel('# videos')
    ax.set_title(f'{name} — frame count distribution')
    ax.legend(fontsize=8)
    print(f'{name}  min={fc.min()}  median={int(np.median(fc))}  '
          f'mean={fc.mean():.1f}  max={fc.max()}  '
          f'shorter_than_64: {(fc<64).sum()} ({(fc<64).mean():.1%})')

plt.tight_layout()
plt.show()

In [ ]:
# Per-class frame count stats for Gym99 (top 20 by sample count)
top20_classes = [c for c, _ in Counter(labels99.tolist()).most_common(20)]

class_fc = {c: frames99[labels99 == c] for c in top20_classes}
means    = [class_fc[c].mean() for c in top20_classes]
stds     = [class_fc[c].std()  for c in top20_classes]

fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(len(top20_classes))
ax.bar(x, means, color='coral', alpha=0.8)
ax.errorbar(x, means, yerr=stds, fmt='none', color='black', capsize=3, linewidth=1)
ax.axhline(64, color='red', linestyle='--', linewidth=1, label='TARGET=64')
ax.set_xticks(x)
ax.set_xticklabels([str(c) for c in top20_classes], rotation=45)
ax.set_ylabel('Mean frame count ± std')
ax.set_title('Gym99 — per-class frame count (top 20 classes by sample count)')
ax.legend()
plt.tight_layout()
plt.show()

# CV (coefficient of variation) of frame counts across classes
class_means288 = np.array([frames288[labels288 == c].mean() for c in np.unique(labels288)])
class_means99  = np.array([frames99[labels99 == c].mean()   for c in np.unique(labels99)])
print(f'Gym288 between-class CV of mean frame count: {class_means288.std()/class_means288.mean():.3f}')
print(f'Gym99  between-class CV of mean frame count: {class_means99.std()/class_means99.mean():.3f}')
print('(Higher CV → frame length varies a lot between action types)')

---
## 5. Skeleton Coordinate Analysis

In [ ]:
# Coordinate space check
sample_ann = raw288['annotations'][0]
kp = np.asarray(sample_ann['keypoint'], dtype=np.float32)
if kp.ndim == 4:
    kp = kp[0]  # (1,T,J,2) → (T,J,2)

print(f'Keypoint shape: {kp.shape}  (T, J, 2)')
print(f'X  min={kp[:,:,0].min():.1f}  max={kp[:,:,0].max():.1f}  '
      f'mean={kp[:,:,0].mean():.1f}')
print(f'Y  min={kp[:,:,1].min():.1f}  max={kp[:,:,1].max():.1f}  '
      f'mean={kp[:,:,1].mean():.1f}')

space = 'pixel space' if kp.max() > 2 else 'normalized [0,1]'
print(f'Coordinate space: {space}')

if 'img_shape' in sample_ann:
    print(f'Image shape: {sample_ann["img_shape"]}')

In [ ]:
# Per-joint 2D position heatmaps (from raw Gym99 annotations)
# Collect all keypoints across all samples
N_SAMPLE = min(3000, len(anns99))   # subsample for speed
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(anns99), N_SAMPLE, replace=False)

all_kpts = []  # list of (T, 17, 2)
for i in sample_idx:
    kp = np.asarray(anns99[i]['keypoint'], dtype=np.float32)
    if kp.ndim == 4:
        kp = kp[0]
    all_kpts.append(kp.reshape(-1, 17, 2))  # flatten T

all_kpts = np.concatenate(all_kpts, axis=0)  # (N*T, 17, 2)

COCO17_NAMES = [
    'nose','l_eye','r_eye','l_ear','r_ear',
    'l_sho','r_sho','l_elb','r_elb','l_wri','r_wri',
    'l_hip','r_hip','l_kne','r_kne','l_ank','r_ank'
]

fig, axes = plt.subplots(3, 6, figsize=(16, 8))
axes = axes.flatten()

for j in range(17):
    x = all_kpts[:, j, 0]
    y = all_kpts[:, j, 1]
    # Filter out zero/invalid (occluded)
    valid = (x > 0) & (y > 0)
    ax = axes[j]
    ax.hist2d(x[valid], y[valid], bins=40, cmap='hot')
    ax.invert_yaxis()
    ax.set_title(COCO17_NAMES[j], fontsize=7)
    ax.axis('off')

for j in range(17, len(axes)):
    axes[j].axis('off')

fig.suptitle(f'Per-joint 2D position heatmap (Gym99, {N_SAMPLE} samples)', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Bounding box size distribution
bbox_widths  = []
bbox_heights = []

for i in sample_idx:
    kp = np.asarray(anns99[i]['keypoint'], dtype=np.float32)
    if kp.ndim == 4:
        kp = kp[0]
    # mid frame
    mid = kp[kp.shape[0] // 2]  # (17, 2)
    valid = mid[(mid[:,0] > 0) & (mid[:,1] > 0)]
    if len(valid) > 1:
        bbox_widths.append(valid[:,0].max() - valid[:,0].min())
        bbox_heights.append(valid[:,1].max() - valid[:,1].min())

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].hist(bbox_widths,  bins=50, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Bbox width (pixels)');  axes[0].set_title('Skeleton bounding box width')
axes[1].hist(bbox_heights, bins=50, color='coral',     edgecolor='white')
axes[1].set_xlabel('Bbox height (pixels)'); axes[1].set_title('Skeleton bounding box height')
plt.tight_layout(); plt.show()

print(f'Width   mean={np.mean(bbox_widths):.1f}  std={np.std(bbox_widths):.1f}  '
      f'min={np.min(bbox_widths):.0f}  max={np.max(bbox_widths):.0f}')
print(f'Height  mean={np.mean(bbox_heights):.1f}  std={np.std(bbox_heights):.1f}  '
      f'min={np.min(bbox_heights):.0f}  max={np.max(bbox_heights):.0f}')

---
## 6. Skeleton Motion Analysis

In [ ]:
# Per-joint velocity: mean |delta| between consecutive frames
# data99: (N, 2, T, V, 1)  →  use processed tensors (already resampled to 64 frames)

N_MOTION = min(3000, data99.shape[0])
idx_motion = rng.choice(data99.shape[0], N_MOTION, replace=False)
sub = data99[idx_motion]  # (N, 2, T, 18, 1)

# velocity per joint: |x[t+1] - x[t]| + |y[t+1] - y[t]|
dx = np.abs(np.diff(sub[:, 0, :, :, 0], axis=1))  # (N, T-1, 18)
dy = np.abs(np.diff(sub[:, 1, :, :, 0], axis=1))
vel = (dx + dy).mean(axis=(0, 1))  # (18,) mean over samples and time

COCO18_NAMES = COCO17_NAMES + ['center']

fig, ax = plt.subplots(figsize=(12, 4))
colors_v = ['#d65f5f' if v > np.percentile(vel, 75) else
             '#4878cf' if v < np.percentile(vel, 25) else
             'steelblue' for v in vel]
ax.bar(range(18), vel, color=colors_v)
ax.set_xticks(range(18))
ax.set_xticklabels(COCO18_NAMES, rotation=45, ha='right')
ax.set_ylabel('Mean absolute velocity (pixels/frame)')
ax.set_title('Per-joint velocity — Gym99 (red=most active, blue=least active)')
plt.tight_layout(); plt.show()

sorted_joints = np.argsort(vel)[::-1]
print('Most  active joints:', [COCO18_NAMES[j] for j in sorted_joints[:5]])
print('Least active joints:', [COCO18_NAMES[j] for j in sorted_joints[-5:]])

In [ ]:
# Per-sample total motion (sum of velocities across all joints and frames)
total_motion = (dx + dy).mean(axis=(1, 2))  # (N,)
sample_labels_motion = labels99t[idx_motion] if hasattr(labels99t, '__len__') else labels99[idx_motion]

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(total_motion, bins=60, color='steelblue', edgecolor='white')
ax.set_xlabel('Mean per-frame joint displacement (pixels)')
ax.set_ylabel('# samples')
ax.set_title('Gym99 — total skeleton motion per sample')
plt.tight_layout(); plt.show()

print(f'Motion  mean={total_motion.mean():.3f}  std={total_motion.std():.3f}  '
      f'min={total_motion.min():.3f}  max={total_motion.max():.3f}')

In [ ]:
# Most vs least dynamic classes (by mean per-sample motion)
class_motion = {}
for c in np.unique(sample_labels_motion):
    mask = sample_labels_motion == c
    if mask.sum() > 1:
        class_motion[c] = total_motion[mask].mean()

sorted_by_motion = sorted(class_motion.items(), key=lambda x: x[1], reverse=True)
top10    = sorted_by_motion[:10]
bottom10 = sorted_by_motion[-10:]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.barh([str(c) for c, _ in top10], [v for _, v in top10], color='#d65f5f')
ax.set_xlabel('Mean motion')
ax.set_title('10 most dynamic classes')

ax = axes[1]
ax.barh([str(c) for c, _ in bottom10], [v for _, v in bottom10], color='#4878cf')
ax.set_xlabel('Mean motion')
ax.set_title('10 least dynamic classes')

plt.tight_layout(); plt.show()

---
## 7. Class Separability — PCA & t-SNE

In [ ]:
# Feature: mean skeleton position over time → (N, 18*2)
# data99: (N, 2, T, 18, 1)
N_TSNE = min(3000, data99.shape[0])
idx_tsne = rng.choice(data99.shape[0], N_TSNE, replace=False)
sub_tsne = data99[idx_tsne]  # (N, 2, T, 18, 1)

# Mean over time + flatten
feats = sub_tsne[:, :, :, :, 0].mean(axis=2)  # (N, 2, 18)
feats = feats.reshape(N_TSNE, -1)              # (N, 36)
lbls_tsne = labels99t[idx_tsne] if hasattr(labels99t, '__len__') else labels99[idx_tsne]

# Limit to top-20 classes for readability
top20 = [c for c, _ in Counter(lbls_tsne.tolist()).most_common(20)]
mask20 = np.isin(lbls_tsne, top20)
feats20 = feats[mask20]
lbls20  = lbls_tsne[mask20]

# PCA
pca = PCA(n_components=2, random_state=SEED)
pca_coords = pca.fit_transform(feats20)

fig, ax = plt.subplots(figsize=(8, 6))
cmap = plt.get_cmap('tab20')
for i, c in enumerate(top20):
    m = lbls20 == c
    ax.scatter(pca_coords[m, 0], pca_coords[m, 1],
               s=12, alpha=0.5, color=cmap(i / 20), label=str(c))
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.set_title('PCA on mean skeleton (top-20 Gym99 classes)')
ax.legend(ncol=4, fontsize=7, markerscale=2)
plt.tight_layout(); plt.show()
print(f'PCA variance explained: PC1={pca.explained_variance_ratio_[0]:.1%}  '
      f'PC2={pca.explained_variance_ratio_[1]:.1%}  '
      f'cumulative={pca.explained_variance_ratio_.sum():.1%}')

In [ ]:
# t-SNE (takes ~1-2 min)
print('Running t-SNE...')
tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, n_iter=1000)
tsne_coords = tsne.fit_transform(feats20)

fig, ax = plt.subplots(figsize=(8, 6))
for i, c in enumerate(top20):
    m = lbls20 == c
    ax.scatter(tsne_coords[m, 0], tsne_coords[m, 1],
               s=12, alpha=0.5, color=cmap(i / 20), label=str(c))
ax.set_title('t-SNE on mean skeleton (top-20 Gym99 classes)')
ax.legend(ncol=4, fontsize=7, markerscale=2)
plt.tight_layout(); plt.show()

---
## 8. Gym288 → Gym99 Mapping Quality

In [ ]:
# Mapping breakdown: direct vs neighbour fallback vs dropped
import urllib.request
from src.config import GYM288_CATEGORIES_URL, GYM99_CATEGORIES_URL
from src.gym99_builder import parse_finegym_categories

text288 = urllib.request.urlopen(GYM288_CATEGORIES_URL).read().decode()
text99  = urllib.request.urlopen(GYM99_CATEGORIES_URL).read().decode()

pairs288 = parse_finegym_categories(text288)
pairs99  = parse_finegym_categories(text99)

clabel288_to_global = dict(pairs288)
global_to_clabel99  = {g: c for c, g in pairs99}

map_288_to_99 = {}
for c288, g in clabel288_to_global.items():
    if g in global_to_clabel99:
        map_288_to_99[c288] = global_to_clabel99[g]

direct = fallback_m1 = fallback_p1 = unmatched = 0
for ann in anns288:
    lbl = int(ann['label'])
    if lbl in map_288_to_99:
        direct += 1
    elif (lbl - 1) in map_288_to_99:
        fallback_m1 += 1
    elif (lbl + 1) in map_288_to_99:
        fallback_p1 += 1
    else:
        unmatched += 1

total = direct + fallback_m1 + fallback_p1 + unmatched

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

labels_pie = ['Direct', 'Fallback −1', 'Fallback +1', 'Dropped']
sizes_pie  = [direct, fallback_m1, fallback_p1, unmatched]
colors_pie = ['steelblue', 'orange', 'coral', 'lightgray']
axes[0].pie(sizes_pie, labels=labels_pie, colors=colors_pie,
            autopct=lambda p: f'{p:.1f}%' if p > 1 else '', startangle=140)
axes[0].set_title('Gym288→Gym99 label mapping breakdown')

# Cumulative sample retention
retained = [direct + fallback_m1 + fallback_p1]
axes[1].bar(['Gym288 (total)', 'Gym99 (mapped)'],
            [total, retained[0]], color=['steelblue', 'coral'])
axes[1].set_ylabel('# samples')
axes[1].set_title(f'Retention: {retained[0]/total:.1%} of Gym288 samples kept')
for i, v in enumerate([total, retained[0]]):
    axes[1].text(i, v + 20, str(v), ha='center', fontsize=10)

plt.tight_layout(); plt.show()

print(f'Direct: {direct} ({direct/total:.1%})  '
      f'Fallback-1: {fallback_m1} ({fallback_m1/total:.1%})  '
      f'Fallback+1: {fallback_p1} ({fallback_p1/total:.1%})  '
      f'Dropped: {unmatched} ({unmatched/total:.1%})')

In [ ]:
# Per Gym99 class: how many samples came via direct vs fallback match
class99_direct   = Counter()
class99_fallback = Counter()

for ann in anns288:
    lbl = int(ann['label'])
    if lbl in map_288_to_99:
        class99_direct[map_288_to_99[lbl]] += 1
    elif (lbl - 1) in map_288_to_99:
        class99_fallback[map_288_to_99[lbl - 1]] += 1
    elif (lbl + 1) in map_288_to_99:
        class99_fallback[map_288_to_99[lbl + 1]] += 1

# Top 20 Gym99 classes by total samples
top20_99 = [c for c, _ in Counter(labels99.tolist()).most_common(20)]
direct_counts   = [class99_direct.get(c, 0)   for c in top20_99]
fallback_counts = [class99_fallback.get(c, 0) for c in top20_99]

x = np.arange(len(top20_99))
fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(x, direct_counts,   label='Direct match',   color='steelblue')
ax.bar(x, fallback_counts, bottom=direct_counts,   label='Fallback match', color='orange', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([str(c) for c in top20_99], rotation=45)
ax.set_ylabel('# samples')
ax.set_title('Gym99 top-20 classes — direct vs fallback sample origin')
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Sample count before (Gym288) vs after (Gym99) remapping
# For each Gym99 class, sum up how many Gym288 samples map to it
gym99_class_counts = Counter(labels99.tolist())

# Gym99 class sizes sorted
gym99_sorted = sorted(gym99_class_counts.items(), key=lambda x: x[1], reverse=True)
class_ids    = [c for c, _ in gym99_sorted]
class_sizes  = [v for _, v in gym99_sorted]

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(class_sizes)), class_sizes, color='coral', width=1.0)
ax.axhline(np.mean(class_sizes), color='black', linestyle='--',
           label=f'mean={np.mean(class_sizes):.0f}')
ax.set_xlabel('Gym99 class rank (sorted)')
ax.set_ylabel('Samples')
ax.set_title('Gym99 — final class distribution after remapping')
ax.legend()
plt.tight_layout(); plt.show()

print(f'Gym99 class size  min={min(class_sizes)}  max={max(class_sizes)}  '
      f'mean={np.mean(class_sizes):.1f}  Gini={gini(class_sizes):.3f}')

---
## 9. Bbox Normalization Effect

In [ ]:
from src.skeleton_utils import bbox_normalize

# Coordinate distributions before vs after for x and y channels
N_NORM = min(2000, data99.shape[0])
idx_norm = rng.choice(data99.shape[0], N_NORM, replace=False)
sub_raw  = data99[idx_norm]           # (N, 2, T, 18, 1)
sub_norm = bbox_normalize(sub_raw)    # same shape

fig, axes = plt.subplots(2, 2, figsize=(12, 7))

for ch, (ch_name, color) in enumerate([('X', 'steelblue'), ('Y', 'coral')]):
    raw_vals  = sub_raw[:, ch].flatten()
    norm_vals = sub_norm[:, ch].flatten()

    ax = axes[ch, 0]
    ax.hist(raw_vals, bins=80, color=color, edgecolor='none', alpha=0.8)
    ax.set_title(f'{ch_name} coordinates — raw (pixel space)')
    ax.set_xlabel('Coordinate value')
    ax.set_ylabel('Count')

    ax = axes[ch, 1]
    ax.hist(norm_vals, bins=80, color=color, edgecolor='none', alpha=0.8)
    ax.set_title(f'{ch_name} coordinates — after bbox_normalize')
    ax.set_xlabel('Coordinate value [0, 1]')

plt.suptitle('Coordinate distribution: raw vs bbox-normalized', fontsize=11)
plt.tight_layout()
plt.show()

for ch, ch_name in [(0, 'X'), (1, 'Y')]:
    r = sub_raw[:, ch].flatten()
    n = sub_norm[:, ch].flatten()
    print(f'{ch_name}  raw:  mean={r.mean():.1f}  std={r.std():.1f}  '
          f'[{r.min():.0f}, {r.max():.0f}]')
    print(f'{ch_name}  norm: mean={n.mean():.3f}  std={n.std():.3f}  '
          f'[{n.min():.3f}, {n.max():.3f}]')

In [ ]:
# Visual comparison: 4 random skeletons before vs after bbox norm
from src.joint_specs import get_joint_spec

spec = get_joint_spec('coco18')

def draw_skeleton_xy(ax, xy, bone_pairs, color='steelblue', title=''):
    for (p, c) in bone_pairs:
        ax.plot([xy[p, 0], xy[c, 0]], [xy[p, 1], xy[c, 1]], color=color, lw=1.5)
    ax.scatter(xy[:, 0], xy[:, 1], s=15, color=color, zorder=5)
    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.axis('off')
    ax.set_title(title, fontsize=7)

N_SHOW = 4
show_idx = rng.choice(N_NORM, N_SHOW, replace=False)
mid_t    = sub_raw.shape[2] // 2

fig, axes = plt.subplots(2, N_SHOW, figsize=(N_SHOW * 3, 6))

for col, si in enumerate(show_idx):
    raw_xy  = np.stack([sub_raw[si,  0, mid_t, :, 0],
                        sub_raw[si,  1, mid_t, :, 0]], axis=1)  # (18, 2)
    norm_xy = np.stack([sub_norm[si, 0, mid_t, :, 0],
                        sub_norm[si, 1, mid_t, :, 0]], axis=1)

    draw_skeleton_xy(axes[0, col], raw_xy,  spec.bone_pairs,
                     color='steelblue', title=f'Raw #{si}')
    draw_skeleton_xy(axes[1, col], norm_xy, spec.bone_pairs,
                     color='coral',     title=f'Norm #{si}')

fig.suptitle('Skeleton visualization: raw (top) vs bbox-normalized (bottom)', fontsize=10)
plt.tight_layout()
plt.show()